# 08 - Evaluacion De Modelos

## Objetivo

Comparar los modelos entrenados para `churn_t_plus_1`, revisar el tradeoff entre detectar churners y mantener precision, y analizar errores por segmentos de negocio.

## Preguntas

1. Que modelo gana segun el ranking automatico.
2. Que modelo parece mas util para retencion.
3. Donde se concentran falsos negativos y falsos positivos.
4. Que deberia mejorarse en la siguiente iteracion.


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, display
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
pd.options.display.float_format = '{:,.4f}'.format

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELING = PROJECT_ROOT / 'data' / 'modeling'
REPORTS_MODELS = PROJECT_ROOT / 'reports' / 'models'
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
MODELS = PROJECT_ROOT / 'models'
TARGET = 'churn_t_plus_1'


In [ ]:
metrics = pd.read_csv(REPORTS_MODELS / 'model_metrics.csv')
ranking = pd.read_csv(REPORTS_MODELS / 'model_ranking.csv')
summary = json.loads((REPORTS_MODELS / 'training_summary.json').read_text(encoding='utf-8'))

display(ranking)
display(pd.DataFrame([summary]))


In [ ]:
plot_metrics = ranking.melt(id_vars=['model', 'rank'], value_vars=['pr_auc_test', 'recall_test', 'f1_test', 'roc_auc_test'], var_name='metric', value_name='value')
plt.figure(figsize=(11, 5))
sns.barplot(data=plot_metrics, x='model', y='value', hue='metric')
plt.title('Comparativa de modelos en test temporal')
plt.xlabel('Modelo')
plt.ylabel('Valor')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
roc_path = FIGURES / 'roc_curves.png'
pr_path = FIGURES / 'pr_curves.png'
if roc_path.exists():
    display(Image(filename=str(roc_path)))
if pr_path.exists():
    display(Image(filename=str(pr_path)))


In [ ]:
test_metrics = metrics[metrics['split'] == 'test_temporal'].copy()
cm_rows = []
for _, row in test_metrics.iterrows():
    cm = np.array(json.loads(row['confusion_matrix']))
    cm_rows.append({'model': row['model'], 'tn': cm[0,0], 'fp': cm[0,1], 'fn': cm[1,0], 'tp': cm[1,1]})
cm_df = pd.DataFrame(cm_rows)
display(cm_df)

fig, axes = plt.subplots(1, len(cm_df), figsize=(4 * len(cm_df), 3.5))
if len(cm_df) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, cm_df.iterrows()):
    cm = np.array([[row['tn'], row['fp']], [row['fn'], row['tp']]])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(row['model'])
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
plt.tight_layout()
plt.show()


In [ ]:
# Reconstruccion del test temporal y predicciones del modelo ganador serializado.
df = pd.read_csv(MODELING / 'churn_modeling_dataset.csv', parse_dates=['fecha'])
months = sorted(df['fecha'].dropna().unique())
cutoff = months[-6]
test_df = df[df['fecha'] >= cutoff].copy()

model = joblib.load(MODELS / 'best_model.joblib')
X_test = test_df.drop(columns=[TARGET, 'cliente_id', 'fecha'])
y_test = test_df[TARGET].astype(int)
test_df['proba_churn'] = model.predict_proba(X_test)[:, 1]
test_df['pred_churn_05'] = (test_df['proba_churn'] >= 0.5).astype(int)

print('Modelo cargado:', summary['best_model'])
print('Filas test:', len(test_df))
print('Tasa churn test:', y_test.mean())


In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(data=test_df, x='proba_churn', hue=TARGET, bins=50, common_norm=False, stat='density')
plt.title('Distribucion de probabilidades - modelo ganador')
plt.xlabel('Probabilidad estimada de churn')
plt.tight_layout()
plt.show()


In [ ]:
thresholds = np.linspace(0.01, 0.50, 50)
threshold_rows = []
for thr in thresholds:
    pred = (test_df['proba_churn'] >= thr).astype(int)
    threshold_rows.append({
        'threshold': thr,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'clientes_alertados': int(pred.sum()),
    })
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df.sort_values('f1', ascending=False).head(10))

plt.figure(figsize=(9, 4))
sns.lineplot(data=threshold_df, x='threshold', y='precision', label='Precision')
sns.lineplot(data=threshold_df, x='threshold', y='recall', label='Recall')
sns.lineplot(data=threshold_df, x='threshold', y='f1', label='F1')
plt.title('Tradeoff por threshold - modelo ganador')
plt.tight_layout()
plt.show()


In [ ]:
def segment_metrics(data, segment_col):
    rows = []
    for value, tmp in data.groupby(segment_col, dropna=False):
        if len(tmp) < 50:
            continue
        rows.append({
            'segmento': segment_col,
            'valor': value,
            'n': len(tmp),
            'churn_rate': tmp[TARGET].mean(),
            'proba_media': tmp['proba_churn'].mean(),
            'recall_05': recall_score(tmp[TARGET], tmp['pred_churn_05'], zero_division=0),
            'precision_05': precision_score(tmp[TARGET], tmp['pred_churn_05'], zero_division=0),
        })
    return pd.DataFrame(rows)

segments = []
for col in ['tipo_plan', 'tipo_zona', 'descuento_activo']:
    segments.append(segment_metrics(test_df, col))

# Segmentos derivados de actividad de soporte e impago.
test_df['impago_segmento'] = np.where(test_df.get('fact_impago_flag', 0).fillna(0) >= 1, 'impago_mes', 'sin_impago_mes')
test_df['soporte_segmento'] = pd.cut(test_df.get('soporte_contactos', pd.Series(0, index=test_df.index)).fillna(0), bins=[-1, 0, 2, 5, 999], labels=['0', '1-2', '3-5', '6+'])
segments.append(segment_metrics(test_df, 'impago_segmento'))
segments.append(segment_metrics(test_df, 'soporte_segmento'))
segment_df = pd.concat(segments, ignore_index=True)
display(segment_df.sort_values(['segmento', 'churn_rate'], ascending=[True, False]))


In [ ]:
errors = test_df.copy()
errors['error_tipo'] = np.select(
    [
        (errors[TARGET] == 1) & (errors['pred_churn_05'] == 1),
        (errors[TARGET] == 1) & (errors['pred_churn_05'] == 0),
        (errors[TARGET] == 0) & (errors['pred_churn_05'] == 1),
    ],
    ['TP', 'FN', 'FP'],
    default='TN'
)
error_counts = errors['error_tipo'].value_counts().rename_axis('error_tipo').reset_index(name='n')
display(error_counts)

plt.figure(figsize=(7, 4))
sns.countplot(data=errors, x='error_tipo', order=['TP', 'FN', 'FP', 'TN'])
plt.title('Tipos de acierto/error con threshold 0.5')
plt.tight_layout()
plt.show()


## Conclusiones

- El ranking automatico selecciona el modelo con mayor PR-AUC, pero el umbral 0.5 puede ser demasiado conservador para churn.
- En un caso de retencion, detectar churners suele importar mas que maximizar accuracy.
- La siguiente iteracion debe ajustar threshold y definir un coste de negocio para falsos negativos y falsos positivos.

## Decisiones Para La Siguiente Iteracion

- Comparar modelos con threshold optimizado, no solo con 0.5.
- Establecer un recall minimo aceptable para acciones de retencion.
- Analizar falsos negativos por plan, zona, impago y soporte.
- Probar aportacion incremental por familias de variables: cliente, facturacion, red, soporte y encuestas.
